# W02 — Get Data การรวบรวมและนำเข้าข้อมูล

## เป้าหมาย
- ปฏิบัติการต่อเนื่องจากสัปดาห์ที่ 1: คัดกรองและจัดเตรียมข้อมูล (ETL) รายฟังก์ชันบนชุดข้อมูล nvc_tech_store.csv ได้
- ใช้ฟังก์ชันตรวจสอบข้อมูลซ้ำซ้อน (duplicated) ตัดช่องว่างส่วนเกิน (str.strip) จัดการค่าว่าง (dropna) คัดกรองขอบเขตข้อมูล (between) และส่งออกไฟล์ (to_csv) ได้
- นำเข้าไฟล์ Excel หลายแผ่นงาน (pd.read_excel) และจัดทำตารางพจนานุกรมข้อมูล (Data Dictionary) ได้อย่างถูกต้อง

## เตรียมไฟล์
- nvc_tech_store.csv
- students.csv
- nvc_store_sales.xlsx

อัปโหลดไฟล์ผ่าน Files ใน Colab ผู้สอนและผู้เรียนเขียนคำสั่งพร้อมกันทีละขั้น ใช้หนึ่งเซลล์ต่อขั้น และตรวจผลก่อนทำต่อ

## โครงสร้างข้อมูล
- nvc_tech_store.csv: 36 แถว 7 คอลัมน์ — order_id, customer_phone, product_name, category, quantity, total_price, payment_status
- students.csv: 30 แถว 5 คอลัมน์ — id, name, gender, major, score
- nvc_store_sales.xlsx / Products: 6 แถว 5 คอลัมน์ — product_id, product_name, category, cost_price, selling_price
- nvc_store_sales.xlsx / Daily_Sales: 15 แถว 5 คอลัมน์ — order_id, product_id, quantity, payment_method, sale_date


## ขั้นที่ 1: [ทบทวน Extract] นำเข้าไลบรารี pandas และอ่านไฟล์ nvc_tech_store.csv

เตรียมไลบรารี pandas และอ่านตารางข้อมูลต้นฉบับ nvc_tech_store.csv พร้อมกำหนดชนิดข้อมูลหมายเลขโทรศัพท์ต่อเนื่องจากสัปดาห์ที่ 1

```python
import pandas as pd
df = pd.read_csv('nvc_tech_store.csv', dtype={'customer_phone': 'string'})
df.shape
```

- `import pandas as pd`: นำเข้าไลบรารี pandas ในชื่อย่อ pd
- `pd.read_csv('nvc_tech_store.csv', dtype={'customer_phone': 'string'})`: อ่านไฟล์ข้อมูลและกำหนดให้คอลัมน์หมายเลขโทรศัพท์เป็นข้อความ
- `df.shape`: ยืนยันขนาดตารางข้อมูลต้นฉบับ (36, 7)

เขียนคำสั่งและกด Shift + Enter ก่อนอ่านผลด้านล่าง


### การตรวจผล

```text
(36, 7)
```

ตารางข้อมูลต้นฉบับขนาด 36 แถว 7 คอลัมน์ พร้อมสำหรับการตรวจสอบและแปลงข้อมูลในขั้นตอนถัดไป

**เมื่อเกิดข้อผิดพลาด:** ตรวจสอบว่าได้อัปโหลดไฟล์ nvc_tech_store.csv เข้าสู่สภาพแวดล้อมการทำงาน (Runtime) ปัจจุบันแล้ว

**ตรวจความเข้าใจ:** ตารางข้อมูลต้นฉบับก่อนเริ่มกระบวนการ ETL มีจำนวนแถวและคอลัมน์เท่าใด


## ขั้นที่ 2: [ETL: การตรวจสอบปัญหา] การคัดกรองแถวที่มีค่าว่างด้วยเงื่อนไข (Boolean Indexing)

ระบุเงื่อนไข df['total_price'].isna() ภายในเครื่องหมายวงเล็บเหลี่ยม df[...] เพื่อคัดกรองเฉพาะแถวที่ไม่มีข้อมูลยอดเงินรวมออกมาตรวจสอบ

```python
df[df['total_price'].isna()]
```

- `df[...]`: การคัดกรองตารางเพื่อแสดงเฉพาะแถวที่เป็นจริงตามเงื่อนไข
- `df['total_price'].isna()`: เงื่อนไขสำหรับตรวจสอบแถวที่คอลัมน์ total_price มีค่าว่าง

เขียนคำสั่งและกด Shift + Enter ก่อนอ่านผลด้านล่าง


### การตรวจผล

| ดัชนี | order_id | customer_phone | product_name | category | quantity | total_price | payment_status |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 7 | ORD-107 | 0856677807 | เว็บแคม Full HD 1080p | อุปกรณ์ต่อพ่วง | 1.0 | ว่าง | รอชำระ |
| 26 | ORD-125 | 0841122325 | สายแลน Cat6 5 เมตร | สายและหน่วยความจำ | 3.0 | ว่าง | ชำระแล้ว |

พบรายการคำสั่งซื้อที่ไม่มีข้อมูลยอดเงินรวมจำนวน 2 รายการ ได้แก่ รหัส ORD-107 (เว็บแคม Full HD 1080p) และรหัส ORD-125 (สายแลน Cat6 5 เมตร)

**เมื่อเกิดข้อผิดพลาด:** ต้องระบุเครื่องหมายวงเล็บเหลี่ยมให้ครบทั้งส่วนของการคัดกรองตาราง df[...] และการอ้างอิงชื่อคอลัมน์ df['total_price']

**ตรวจความเข้าใจ:** รายการคำสั่งซื้อรหัสใดบ้างที่ไม่มีข้อมูลในคอลัมน์ยอดเงินรวม (total_price)


## ขั้นที่ 3: [ETL: การตรวจสอบปัญหา] .duplicated().sum() นับจำนวนแถวข้อมูลที่ซ้ำซ้อน

เมธอด .duplicated() ทำหน้าที่ตรวจสอบแถวที่มีข้อมูลซ้ำกับแถวก่อนหน้าในทุกคอลัมน์ และใช้ .sum() เพื่อรวมจำนวนแถวที่ซ้ำซ้อนทั้งหมด

```python
df.duplicated().sum()
```

- `df.duplicated()`: เมธอดตรวจสอบแถวที่มีข้อมูลซ้ำกับแถวก่อนหน้าทุกคอลัมน์
- `.sum()`: เมธอดรวมจำนวนแถวที่ตรวจพบว่าเป็นข้อมูลซ้ำ

เขียนคำสั่งและกด Shift + Enter ก่อนอ่านผลด้านล่าง


### การตรวจผล

```text
np.int64(2)
```

ได้ผลลัพธ์เท่ากับ 2 แสดงว่ามีรายการคำสั่งซื้อที่ถูกบันทึกซ้ำซ้อนเข้ามาเกินจำนวนจริง 2 แถว

**เมื่อเกิดข้อผิดพลาด:** หากไม่ตรวจสอบและจัดการแถวข้อมูลที่ซ้ำซ้อน จะส่งผลให้การคำนวณยอดจำหน่ายรวมและจำนวนคำสั่งซื้อสูงกว่าความเป็นจริง

**ตรวจความเข้าใจ:** ในชุดข้อมูลนี้มีแถวข้อมูลที่ถูกบันทึกซ้ำซ้อนจำนวนกี่แถว


## ขั้นที่ 4: [ETL: การตรวจสอบปัญหา] การแสดงแถวข้อมูลที่ซ้ำซ้อนทั้งหมดด้วย duplicated(keep=False)

กำหนดพารามิเตอร์ keep=False เพื่อแสดงทั้งแถวข้อมูลต้นฉบับและแถวที่บันทึกซ้ำซ้อนสำหรับใช้เปรียบเทียบและยืนยันความถูกต้อง

```python
df[df.duplicated(keep=False)]
```

- `df[...]`: การคัดกรองตารางเพื่อแสดงเฉพาะแถวที่ตรงตามเงื่อนไข
- `df.duplicated(keep=False)`: เงื่อนไขที่กำหนดให้แสดงทุกแถวที่มีข้อมูลซ้ำกัน (รวมแถวแรกและแถวที่ซ้ำ)

เขียนคำสั่งและกด Shift + Enter ก่อนอ่านผลด้านล่าง


### การตรวจผล

| ดัชนี | order_id | customer_phone | product_name | category | quantity | total_price | payment_status |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 4 | ORD-105 | 0823344505 | แฟลชไดรฟ์ 64GB USB3 | สายและหน่วยความจำ | 1.0 | 250.0 | ชำระแล้ว |
| 5 | ORD-105 | 0823344505 | แฟลชไดรฟ์ 64GB USB3 | สายและหน่วยความจำ | 1.0 | 250.0 | ชำระแล้ว |
| 18 | ORD-118 | 0836655418 | เมาส์ไร้สาย NVC Pro | อุปกรณ์ต่อพ่วง | 2.0 | 580.0 | ชำระแล้ว |
| 19 | ORD-118 | 0836655418 | เมาส์ไร้สาย NVC Pro | อุปกรณ์ต่อพ่วง | 2.0 | 580.0 | ชำระแล้ว |

ปรากฏรายการที่ซ้ำซ้อนกันจำนวน 4 แถว (2 คู่รายการ) ได้แก่ รหัส ORD-105 (ดัชนี 4 และ 5) และรหัส ORD-118 (ดัชนี 18 และ 19) ซึ่งมีข้อมูลตรงกันทุกคอลัมน์

**เมื่อเกิดข้อผิดพลาด:** หากไม่ระบุพารามิเตอร์ keep=False ระบบจะแสดงเฉพาะแถวที่ซ้ำในลำดับถัดมา (ดัชนี 5 และ 19) โดยไม่แสดงแถวแรกเพื่อเปรียบเทียบ

**ตรวจความเข้าใจ:** รหัสคำสั่งซื้อใดบ้างที่ปรากฏการบันทึกข้อมูลซ้ำซ้อนกัน


## ขั้นที่ 5: [ETL: การตรวจสอบปัญหา] .value_counts() ตรวจสอบความถี่ของหมวดหมู่สินค้าและอักขระช่องว่างส่วนเกิน

เมธอด .value_counts() ทำหน้าที่นับจำนวนความถี่ของข้อมูลแต่ละค่าในคอลัมน์ ช่วยในการตรวจสอบความสม่ำเสมอของการสะกดคำและช่องว่างส่วนเกิน

```python
df['category'].value_counts()
```

- `df['category']`: การอ้างอิงคอลัมน์หมวดหมู่สินค้า
- `.value_counts()`: เมธอดนับจำนวนความถี่ของข้อมูลแต่ละกลุ่มเรียงจากมากไปน้อย

เขียนคำสั่งและกด Shift + Enter ก่อนอ่านผลด้านล่าง


### การตรวจผล

```text
category
สายและหน่วยความจำ    14
อุปกรณ์ต่อพ่วง       13
อุปกรณ์เสียง          7
อุปกรณ์ต่อพ่วง        2
```

พบว่าหมวดหมู่ 'อุปกรณ์ต่อพ่วง' ถูกแยกนับเป็น 2 กลุ่ม คือ 13 รายการ และ 2 รายการ เนื่องจากมีอักขระช่องว่างต่อท้ายข้อความ ('อุปกรณ์ต่อพ่วง ')

**เมื่อเกิดข้อผิดพลาด:** ระบบประมวลผลถือว่าข้อความ 'อุปกรณ์ต่อพ่วง' และ 'อุปกรณ์ต่อพ่วง ' (มีช่องว่างต่อท้าย) เป็นข้อมูลคนละกลุ่มกัน

**ตรวจความเข้าใจ:** เหตุใดหมวดหมู่ 'อุปกรณ์ต่อพ่วง' จึงถูกแยกแสดงผลเป็น 2 รายการ (13 แถว และ 2 แถว)


## ขั้นที่ 6: [ETL: Transform 1] .str.strip() ตัดอักขระช่องว่างส่วนเกินบริเวณหน้าและหลังข้อความ

ใช้เมธอดจัดการข้อความ .str.strip() เพื่อลบอักขระช่องว่าง (Whitespace) บริเวณด้านหน้าและด้านหลังของข้อความ แล้วบันทึกค่ากลับเข้าสู่คอลัมน์เดิม

```python
df['category'] = df['category'].str.strip()
df['category'].value_counts()
```

- `df['category'] =`: บันทึกผลลัพธ์ที่ผ่านการตัดช่องว่างแล้วกลับเข้าสู่คอลัมน์ category
- `df['category'].str.strip()`: เรียกใช้ชุดคำสั่งข้อความ (.str) และเมธอดตัดช่องว่างส่วนเกิน (.strip())
- `df['category'].value_counts()`: ตรวจสอบความถี่ซ้ำอีกครั้งเพื่อยืนยันการรวมกลุ่มของหมวดหมู่สินค้า

เขียนคำสั่งและกด Shift + Enter ก่อนอ่านผลด้านล่าง


### การตรวจผล

```text
category
อุปกรณ์ต่อพ่วง       15
สายและหน่วยความจำ    14
อุปกรณ์เสียง          7
```

หมวดหมู่ 'อุปกรณ์ต่อพ่วง' ถูกรวมเป็นกลุ่มเดียวกันจำนวน 15 รายการ ส่งผลให้ตารางมีหมวดหมู่สินค้า 3 หมวดหมู่ตามโครงสร้างมาตรฐาน

**เมื่อเกิดข้อผิดพลาด:** หากไม่ระบุคำสั่งกำหนดค่า df['category'] = ด้านหน้า ผลลัพธ์จากการตัดช่องว่างจะไม่ถูกบันทึกทับลงในตาราง

**ตรวจความเข้าใจ:** ภายหลังการใช้คำสั่ง .str.strip() หมวดหมู่ 'อุปกรณ์ต่อพ่วง' มีจำนวนรายการรวมทั้งสิ้นเท่าใด


## ขั้นที่ 7: [ETL: Transform 2] .drop_duplicates() ลบแถวข้อมูลที่ซ้ำซ้อนและแยกจัดเก็บในตัวแปร df_clean

ใช้เมธอด .drop_duplicates() เพื่อตัดแถวข้อมูลที่ซ้ำซ้อนออกโดยคงแถวแรกไว้ และจัดเก็บลงในตัวแปรตารางใหม่ชื่อ df_clean เพื่อรักษาตารางต้นฉบับไว้

```python
df_clean = df.drop_duplicates().copy()
df_clean.shape
```

- `df_clean =`: กำหนดตัวแปรตารางใหม่สำหรับจัดเก็บข้อมูลที่ผ่านการทำความสะอาด
- `df.drop_duplicates().copy()`: เมธอดลบแถวข้อมูลที่ซ้ำซ้อนและสร้างสำเนาตาราง
- `df_clean.shape`: ตรวจสอบขนาดตารางภายหลังการลบแถวข้อมูลที่ซ้ำซ้อน

เขียนคำสั่งและกด Shift + Enter ก่อนอ่านผลด้านล่าง


### การตรวจผล

```text
(34, 7)
```

ขนาดของตารางลดลงจาก (36, 7) เหลือ (34, 7) เนื่องจากแถวที่ซ้ำซ้อนของรหัส ORD-105 และ ORD-118 ถูกลบออกจำนวน 2 แถว

**เมื่อเกิดข้อผิดพลาด:** ในการจัดเตรียมข้อมูลควรสร้างตัวแปรใหม่ (เช่น df_clean) เพื่อเก็บรักษาตารางข้อมูลต้นฉบับ (df) ไว้สำหรับตรวจสอบเปรียบเทียบเสมอ

**ตรวจความเข้าใจ:** เหตุใดจำนวนแถวของตาราง df_clean จึงลดลงจาก 36 แถว เหลือ 34 แถว


## ขั้นที่ 8: [ETL: Transform 3] .dropna(subset=...) คัดกรองแถวที่มีค่าว่างในคอลัมน์สำคัญออก

ใช้เมธอด .dropna() ร่วมกับพารามิเตอร์ subset เพื่อลบเฉพาะแถวที่ไม่มีข้อมูลจำนวนชิ้น (quantity) หรือยอดเงินรวม (total_price) โดยคงแถวที่ขาดเฉพาะหมายเลขโทรศัพท์ไว้

```python
df_clean = df_clean.dropna(subset=['quantity', 'total_price'])
df_clean.shape
```

- `df_clean.dropna(...)`: เมธอดสำหรับลบแถวที่ปรากฏค่าว่าง (Missing Values)
- `subset=['quantity', 'total_price']`: พารามิเตอร์กำหนดให้ตรวจสอบค่าว่างเฉพาะในคอลัมน์ quantity และ total_price
- `df_clean.shape`: ตรวจสอบขนาดตารางภายหลังการคัดกรองแถวที่มีค่าว่าง

เขียนคำสั่งและกด Shift + Enter ก่อนอ่านผลด้านล่าง


### การตรวจผล

```text
(31, 7)
```

ขนาดของตารางลดลงจาก (34, 7) เหลือ (31, 7) โดยตัดเฉพาะ 3 แถวที่ไม่สามารถคำนวณมูลค่าการซื้อขายได้ ในขณะที่รายการที่ขาดเฉพาะหมายเลขโทรศัพท์ยังคงอยู่ในระบบ

**เมื่อเกิดข้อผิดพลาด:** หากเรียกใช้ df_clean.dropna() โดยไม่ระบุพารามิเตอร์ subset ระบบจะลบรายการ ORD-104 และ ORD-122 ซึ่งมีมูลค่าการซื้อขายจริงแต่ไม่ระบุหมายเลขโทรศัพท์ออกไปด้วย

**ตรวจความเข้าใจ:** การกำหนดพารามิเตอร์ subset=['quantity', 'total_price'] มีข้อดีอย่างไรเมื่อเปรียบเทียบกับการใช้คำสั่ง dropna() ทั้งตาราง


## ขั้นที่ 9: [ETL: Transform 4] .between() คัดกรองข้อมูลให้อยู่ในเกณฑ์ที่กำหนดและเปรียบเทียบค่าเฉลี่ย

ใช้เมธอด .between(1, 20) เพื่อคัดกรองจำนวนชิ้นที่อยู่ในช่วง 1 ถึง 20 ชิ้น และคัดกรองยอดเงินรวมไม่เกิน 10,000 บาท จากนั้นเปรียบเทียบค่าเฉลี่ยก่อนและหลังการจัดเตรียมข้อมูล

```python
df_clean = df_clean[df_clean['quantity'].between(1, 20) & (df_clean['total_price'] <= 10000)]
round(df['total_price'].mean(), 2), round(df_clean['total_price'].mean(), 2)
```

- `df_clean['quantity'].between(1, 20)`: เมธอดตรวจสอบว่าค่าในคอลัมน์ quantity อยู่ในช่วงตั้งแต่ 1 ถึง 20 หรือไม่
- `& (df_clean['total_price'] <= 10000)`: ตัวดำเนินการเชื่อมเงื่อนไขว่ายอดเงินรวมต่อรายการต้องไม่เกิน 10,000 บาท
- `round(..., 2)`: ฟังก์ชันปัดเศษทศนิยม 2 ตำแหน่งเพื่อเปรียบเทียบค่าเฉลี่ยของตาราง df และ df_clean

เขียนคำสั่งและกด Shift + Enter ก่อนอ่านผลด้านล่าง


### การตรวจผล

```text
(np.float64(3613.82), np.float64(754.83))
```

ผลการเปรียบเทียบแสดงให้เห็นว่า ค่าเฉลี่ยยอดสั่งซื้อก่อนการทำ ETL เท่ากับ 3,677.94 บาท (เนื่องจากมีค่าผิดปกติ 99,000 บาท) แต่ภายหลังการทำ ETL ค่าเฉลี่ยเท่ากับ 823.79 บาท ซึ่งสะท้อนมูลค่าการจำหน่ายจริง

**เมื่อเกิดข้อผิดพลาด:** การเชื่อมหลายเงื่อนไขใน pandas ต้องใช้เครื่องหมาย & และใส่วงเล็บครอบแต่ละเงื่อนไขย่อยให้ครบถ้วน

**ตรวจความเข้าใจ:** ค่าเฉลี่ยยอดสั่งซื้อต่อรายการก่อนและหลังการทำ ETL แตกต่างกันเท่าใด และมีสาเหตุมาจากปัจจัยใด


## ขั้นที่ 10: [ETL: Load] .to_csv() บันทึกตารางข้อมูลที่ผ่านการจัดเตรียมออกเป็นไฟล์ CSV ใหม่

ขั้นตอนสุดท้ายของกระบวนการ ETL (Load): ใช้เมธอด .to_csv() เพื่อส่งออกตารางข้อมูล df_clean (จำนวน 29 แถว) เป็นไฟล์ CSV ใหม่โดยไม่บันทึกหมายเลขดัชนี

```python
df_clean.to_csv('nvc_tech_store_clean.csv', index=False)
df_clean.head()
```

- `df_clean.to_csv('nvc_tech_store_clean.csv', ...)`: เมธอดสำหรับส่งออกตารางข้อมูลเป็นไฟล์ CSV ตามชื่อที่กำหนด
- `index=False`: พารามิเตอร์กำหนดไม่ให้บันทึกหมายเลขดัชนีของตารางเป็นคอลัมน์เพิ่มเติมในไฟล์ CSV
- `df_clean.head()`: แสดงข้อมูลตัวอย่าง 5 แถวแรกของตารางที่ผ่านการจัดเตรียมเรียบร้อยแล้ว

เขียนคำสั่งและกด Shift + Enter ก่อนอ่านผลด้านล่าง


### การตรวจผล

| ดัชนี | order_id | customer_phone | product_name | category | quantity | total_price | payment_status |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 0 | ORD-101 | 0812345601 | เมาส์ไร้สาย NVC Pro | อุปกรณ์ต่อพ่วง | 2.0 | 580.0 | ชำระแล้ว |
| 1 | ORD-102 | 0891122302 | คีย์บอร์ดเชิงกล RGB | อุปกรณ์ต่อพ่วง | 1.0 | 1290.0 | ชำระแล้ว |
| 2 | ORD-103 | 0865544303 | แผ่นรองเมาส์ XL | อุปกรณ์ต่อพ่วง | 3.0 | 450.0 | ชำระแล้ว |
| 3 | ORD-104 | ว่าง | สายชาร์จ Type-C Fast | สายและหน่วยความจำ | 2.0 | 380.0 | ชำระแล้ว |
| 4 | ORD-105 | 0823344505 | แฟลชไดรฟ์ 64GB USB3 | สายและหน่วยความจำ | 1.0 | 250.0 | ชำระแล้ว |

ระบบสร้างไฟล์ nvc_tech_store_clean.csv (ขนาด 29 แถว 7 คอลัมน์) ในส่วนจัดเก็บไฟล์ของ Google Colab พร้อมสำหรับการนำไปวิเคราะห์และสร้างทัศนภาพข้อมูล

**เมื่อเกิดข้อผิดพลาด:** หากไม่ระบุพารามิเตอร์ index=False เมื่อนำไฟล์ CSV ดังกล่าวกลับมาเปิดใช้งานใหม่ จะปรากฏคอลัมน์ Unnamed: 0 เพิ่มขึ้นด้านซ้ายสุด

**ตรวจความเข้าใจ:** พารามิเตอร์ index=False ในคำสั่ง .to_csv() มีหน้าที่และผลลัพธ์อย่างไร


## วันอังคาร

ตรวจไฟล์ใน Runtime และเรียกใช้เซลล์วันจันทร์ก่อนเพิ่มคำสั่งใหม่


## ขั้นที่ 11: อ่านแผ่นงานข้อมูลสินค้าจากไฟล์ Excel

อ่านข้อมูลจากแผ่นงาน Products ในไฟล์ Excel และแสดงตารางข้อมูลสินค้า

```python
products = pd.read_excel('nvc_store_sales.xlsx', sheet_name='Products')
products
```

- `pd.read_excel(...)`: ฟังก์ชันสำหรับอ่านไฟล์ Excel
- `'nvc_store_sales.xlsx'`: ชื่อไฟล์ข้อมูล Excel
- `sheet_name='Products'`: พารามิเตอร์ระบุชื่อแผ่นงานข้อมูลสินค้า
- `products`: แสดงตารางข้อมูลสินค้าทั้งหมด 6 แถว

เขียนคำสั่งและกด Shift + Enter ก่อนอ่านผลด้านล่าง


### การตรวจผล

| ดัชนี | product_id | product_name | category | cost_price | selling_price |
| --- | --- | --- | --- | --- | --- |
| 0 | P001 | ชาไทยเย็น | เครื่องดื่มชง | 15 | 30 |
| 1 | P002 | กาแฟโบราณ | เครื่องดื่มชง | 18 | 35 |
| 2 | P003 | ชามะนาว | เครื่องดื่มผลไม้ | 12 | 25 |
| 3 | P004 | โกโก้เข้มข้น | เครื่องดื่มชง | 18 | 35 |
| 4 | P005 | นมสดคาราเมล | เครื่องดื่มนม | 20 | 35 |
| 5 | P006 | ชาเขียวนม | เครื่องดื่มชง | 20 | 40 |

แผ่นงาน Products มีขนาด 6 แถว 5 คอลัมน์ จัดเก็บรหัสสินค้า ชื่อสินค้า หมวดหมู่ ต้นทุน และราคาจำหน่ายต่อหน่วย

**เมื่อเกิดข้อผิดพลาด:** การระบุชื่อแผ่นงานต้องสะกดตัวอักษรพิมพ์ใหญ่และพิมพ์เล็กให้ตรงตามต้นฉบับ เช่น Products

**ตรวจความเข้าใจ:** คอลัมน์ unit_price ในตารางสินค้าหมายถึงข้อมูลใดและมีหน่วยวัดเป็นอะไร


## ขั้นที่ 12: อ่านแผ่นงานรายการจำหน่ายรายวันจากไฟล์ Excel

อ่านข้อมูลจากแผ่นงาน Daily_Sales และแสดงข้อมูลตัวอย่าง 5 แถวแรกของรายการจำหน่าย

```python
sales = pd.read_excel('nvc_store_sales.xlsx', sheet_name='Daily_Sales')
sales.head()
```

- `sales =`: กำหนดตัวแปรสำหรับจัดเก็บตารางรายการจำหน่าย
- `sheet_name='Daily_Sales'`: พารามิเตอร์ระบุชื่อแผ่นงานรายการจำหน่ายรายวัน
- `sales.head()`: แสดงข้อมูลตัวอย่าง 5 แถวแรกของตารางรายการจำหน่าย

เขียนคำสั่งและกด Shift + Enter ก่อนอ่านผลด้านล่าง


### การตรวจผล

| ดัชนี | order_id | product_id | quantity | payment_method | sale_date |
| --- | --- | --- | --- | --- | --- |
| 0 | ORD001 | P001 | 2 | เงินสด | 2026-11-01 |
| 1 | ORD002 | P002 | 1 | พร้อมเพย์ | 2026-11-01 |
| 2 | ORD003 | P001 | 3 | พร้อมเพย์ | 2026-11-01 |
| 3 | ORD004 | P004 | 1 | เงินสด | 2026-11-01 |
| 4 | ORD005 | P003 | 2 | พร้อมเพย์ | 2026-11-01 |

ปรากฏรหัสรายการจำหน่าย วันที่จำหน่าย รหัสสินค้า จำนวนชิ้น และวิธีการชำระเงิน

**เมื่อเกิดข้อผิดพลาด:** หากไม่ระบุพารามิเตอร์ sheet_name ไลบรารี pandas จะอ่านเฉพาะแผ่นงานแรกของไฟล์ Excel เท่านั้น

**ตรวจความเข้าใจ:** คอลัมน์ใดทำหน้าที่เป็นคีย์สำหรับเชื่อมโยงตาราง sales กับตาราง products


## ขั้นที่ 13: ตรวจสอบโครงสร้างของตารางรายการจำหน่าย

ตรวจสอบรายชื่อคอลัมน์ ชนิดข้อมูล และขนาดของตาราง sales

```python
sales.columns.tolist(), sales.dtypes, sales.shape
```

- `sales.columns.tolist()`: แปลงรายชื่อคอลัมน์ให้อยู่ในรูปแบบรายการ (List)
- `sales.dtypes`: ตรวจสอบชนิดข้อมูลของแต่ละคอลัมน์
- `sales.shape`: ตรวจสอบจำนวนแถวและจำนวนคอลัมน์

เขียนคำสั่งและกด Shift + Enter ก่อนอ่านผลด้านล่าง


### การตรวจผล

```text
(['order_id', 'product_id', 'quantity', 'payment_method', 'sale_date'], order_id          object
product_id        object
quantity           int64
payment_method    object
sale_date         object
dtype: object, (15, 5))
```

ตาราง Daily_Sales มีขนาด 15 แถว 5 คอลัมน์ โดยคอลัมน์ quantity เป็นจำนวนเต็มและคอลัมน์อื่นเป็นข้อความ

**เมื่อเกิดข้อผิดพลาด:** คอลัมน์ sale_date ยังมีชนิดข้อมูลเป็นข้อความ จึงควรตรวจสอบรูปแบบวันที่ก่อนนำไปจัดเรียงตามลำดับเวลา

**ตรวจความเข้าใจ:** ตาราง Daily_Sales มีจำนวนแถวและจำนวนคอลัมน์ทั้งหมดเท่าใด


## ขั้นที่ 14: จัดทำตารางพจนานุกรมข้อมูลของ nvc_tech_store.csv

สร้างตารางสรุปรายชื่อคอลัมน์ ชนิดข้อมูล และความหมายของแต่ละคอลัมน์ในตาราง nvc_tech_store

```python
dictionary = pd.DataFrame({
    'column': df.columns,
    'dtype': df.dtypes.astype(str).values,
    'meaning': ['รหัสคำสั่งซื้อ', 'หมายเลขโทรศัพท์ลูกค้า', 'ชื่อสินค้า', 'หมวดหมู่สินค้า', 'จำนวนชิ้น', 'ยอดเงินรวม (บาท)', 'สถานะการชำระเงิน']
})
dictionary
```

- `pd.DataFrame({...})`: สร้างตารางข้อมูลใหม่จากพจนานุกรม (Dictionary)
- `df.columns`: ดึงรายชื่อคอลัมน์ทั้งหมดจากตาราง df
- `df.dtypes.astype(str).values`: แปลงชนิดข้อมูลของแต่ละคอลัมน์เป็นข้อความ

เขียนคำสั่งและกด Shift + Enter ก่อนอ่านผลด้านล่าง


### การตรวจผล

| ดัชนี | column | dtype | meaning |
| --- | --- | --- | --- |
| 0 | order_id | object | รหัสคำสั่งซื้อ |
| 1 | customer_phone | string | หมายเลขโทรศัพท์ลูกค้า |
| 2 | product_name | object | ชื่อสินค้า |
| 3 | category | object | หมวดหมู่สินค้า |
| 4 | quantity | float64 | จำนวนชิ้น |
| 5 | total_price | float64 | ยอดเงินรวม (บาท) |
| 6 | payment_status | object | สถานะการชำระเงิน |

ได้ตารางพจนานุกรมข้อมูลจำนวน 7 แถว 3 คอลัมน์ แสดงชื่อคอลัมน์ ชนิดข้อมูล และความหมายของชุดข้อมูลร้านอุปกรณ์ไอทีอย่างครบถ้วน

**เมื่อเกิดข้อผิดพลาด:** จำนวนคำอธิบายในรายการ meaning ต้องมีจำนวนเท่ากับจำนวนคอลัมน์ของตาราง (7 รายการ)

**ตรวจความเข้าใจ:** การระบุความหมายและหน่วยของคอลัมน์ไว้ในพจนานุกรมข้อมูลมีประโยชน์ต่อผู้วิเคราะห์อย่างไร


## สรุป

อธิบายคำสั่ง ผลลัพธ์ หน่วย และข้อจำกัด บันทึก Notebook สำหรับทบทวน
